# Tema: Lakeflow Jobs, CI/CD y operación

## Objetivos
Construir un DAG, practicar fallo/reparación, parametrizar despliegues y diagnosticar ejecuciones.

## Conceptos importantes para el examen
Dependencias; retries; condiciones y For each; schedule/file arrival/table update; Git folders; Declarative Automation Bundles (antes Asset Bundles); serverless/clásico y costes.

**Dificultad:** Examen · **Tiempo estimado:** 110 min.

El bundle resources/bundle es ejecutable con CLI autenticada y serverless habilitado. Este notebook no crea jobs automáticamente. Los ejercicios combinan código local y acciones concretas en Jobs & Pipelines. Usa un schema distinto por entorno.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_24_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

In [ ]:
print("Variables para CLI:", f"catalog={CATALOG},schema={SCHEMA}")
RUN_JOB_CHECKS = False

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Parámetros de notebook

In [ ]:
dbutils.widgets.text("process_date", "2026-01-01")
process_date = dbutils.widgets.get("process_date")
assert len(process_date)==10
display(employees.withColumn("process_date",F.to_date(F.lit(process_date))))

### 2. Contrato ejecutable de dependencias

In [ ]:
dag = {"bronze": [], "silver": ["bronze"], "gold": ["silver"]}
completed = set()
for task, dependencies in dag.items():
    assert all(d in completed for d in dependencies)
    completed.add(task)
print(dag)
# El scheduler real ejecuta estas dependencias desde databricks.yml.

### 3. Diagnóstico con métricas ficticias
En un job real consulta Run history y las ejecuciones individuales.

In [ ]:
runs = spark.createDataFrame([(1,30,"SUCCESS"),(2,32,"SUCCESS"),(3,31,"SUCCESS"),(4,150,"FAILED")], "run_id INT, seconds INT, status STRING")
display(runs)
print("Baseline:", runs.filter("status='SUCCESS'").agg(F.avg("seconds")).first()[0])

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Valida y despliega el bundle en dev con el catálogo/schema de esta sesión; ejecútalo y comprueba total Gold 780.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Ejecuta el job con fail_silver=true, observa retries y Gold bloqueada. Repara con false y verifica qué tareas vuelven a correr.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Recrea en una copia del job la condición sobre rows de Bronze y el bucle For each sobre dos fechas usando la UI; inspecciona el JSON resultante.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Elige tres triggers y crea un schedule pausado en dev. Explica cuándo usar tareas SQL, dashboard y pipeline.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Promueve el mismo código a test con schema diferente; practica rama, commit y PR desde Git folders sin mezclar datos de dev.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 6
Diagnostica la ejecución lenta/fallida y plantea una comprobación para arranque, librerías y memoria. Consulta optimización predictiva sin activarla.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Los comandos están en resources/bundle/README.md; desplegar no es ejecutar.

**Pista 2:** Repair run evita repetir tareas exitosas innecesariamente.

**Pista 3:** taskValues.rows existe en job_task.py; el For each permite pasar {{input}}.

**Pista 4:** Programado para hora fija; file arrival para landing; table update para dependencia de tablas.

**Pista 5:** Configura target y variables; los cambios de código se revisan antes de promover.

**Pista 6:** Run history, eventos de compute, logs y Spark UI responden preguntas distintas.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
print(f"databricks bundle validate -t dev --var catalog={CATALOG},schema={SCHEMA}")
print(f"databricks bundle deploy -t dev --var catalog={CATALOG},schema={SCHEMA}")
print(f"databricks bundle run -t dev --var catalog={CATALOG},schema={SCHEMA} medallion_job")
if RUN_JOB_CHECKS:
    assert spark.table("job_gold").agg(F.sum("revenue")).first()[0] == 780

### Solución 2

In [ ]:
# UI: Run now con parámetros → fail_silver=true.
# Tras el fallo: Repair run → parámetros fail_silver=false → reparar tareas fallidas y downstream.
# Evidencia: Bronze SUCCESS; Silver FAILED; Gold UPSTREAM_FAILED/bloqueada.
# Tras reparar: Silver/Gold SUCCESS, resultado 780.
if RUN_JOB_CHECKS:
    display(spark.table("job_gold"))

### Solución 3

In [ ]:
import json
condition_task = {"task_key":"has_rows","depends_on":[{"task_key":"bronze"}],
 "condition_task":{"op":"GREATER_THAN","left":"{{tasks.bronze.values.rows}}","right":"0"}}
loop_task = {"task_key":"dates","depends_on":[{"task_key":"gold"}],
 "for_each_task":{"inputs":json.dumps(["2026-01-01","2026-01-02"]),"concurrency":1,
 "task":{"task_key":"process_date","notebook_task":{
 "notebook_path":"RUTA_WORKSPACE_DE_job_task",
 "base_parameters":{"stage":"dates","process_date":"{{input}}"}}}}}
print(json.dumps([condition_task,loop_task],indent=2))
# La configuración ejecutable completa ya está en resources/bundle/databricks.yml.
# UI: Silver depende de has_rows con outcome=true.
# For each invoca job_task.py con stage=dates y process_date={{input}}.
# Ese notebook hace MERGE idempotente de cada fecha en job_dates.
# En la copia del job, usa la ruta real del notebook; conserva catalog/schema como parámetros.
if RUN_JOB_CHECKS:
    assert spark.table("job_dates").count()==2
    display(spark.table("job_dates"))


### Solución 4

In [ ]:
schedule = {"quartz_cron_expression":"0 0 8 * * ?", "timezone_id":"Europe/Madrid", "pause_status":"PAUSED"}
print(json.dumps(schedule,indent=2))
# UI Job → Schedule & Triggers: configura este schedule sin activarlo todavía.
# Alternativas: File arrival sobre ruta autorizada; Table update sobre tabla UC.
# Task types: Notebook para PySpark; SQL query con warehouse; Dashboard para refresco;
# Pipeline task referencia pipeline_id de 22/23 y depende de disponibilidad de la fuente.

### Solución 5

In [ ]:
TEST_SCHEMA = SCHEMA + "_test"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(TEST_SCHEMA)}")
print(f"databricks bundle validate -t test --var catalog={CATALOG},schema={TEST_SCHEMA}")
print(f"databricks bundle deploy -t test --var catalog={CATALOG},schema={TEST_SCHEMA}")
# Workspace → Git folder del repo → nueva rama practice/jobs.
# Edita job_task.py, revisa diff, commit; push y PR cuando decidas publicar tus cambios.
# CI: validación estática + bundle validate; CD: deploy del target aprobado.
# El target cambia configuración; no exige duplicar el código.

### Solución 6

In [ ]:
display(runs.withColumn("slow",F.col("seconds")>60))
display(spark.sql("DESCRIBE DETAIL employees"))
# Arranque: revisar eventos del compute, cuotas/capacidad/políticas.
# Librerías: logs de instalación, versiones y dependencias incompatibles.
# OOM driver: collect/toPandas; OOM executor: particiones, skew, joins/estado.
# Spark UI: shuffle, spill y distribución de tareas; serverless: perfil disponible.
# Optimización predictiva: mantenimiento automático de tablas managed UC,
# depende de habilitación y permisos. Liquid clustering organiza datos por claves.
# Coste: medir tiempo activo y recursos; serverless evita gestión pero no es gratuito.
# No cambies memoria o particiones sin una hipótesis y una medición comparable.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
Silver falla tras Bronze exitosa. ¿Qué acción permite recuperar sin repetir todo?

A. Borrar el catálogo

B. Repair run de tareas afectadas

C. Cambiar el nombre de Gold

D. Ignorar Silver

### Pregunta 2
¿Qué promueve el mismo código con configuración distinta?

A. Copiar manualmente todos los notebooks

B. Cambiar datos a mano

C. Targets y variables de un bundle

D. Eliminar Git

### Pregunta 3
¿Qué evidencia ayuda a identificar spill?

A. Métricas de tareas/stages de Spark

B. El nombre del job

C. Número de usuarios del grupo

D. El README

### Respuestas y explicación
**1. B** — La reparación puede reutilizar tareas exitosas.

**2. C** — Separa código de configuración por entorno.

**3. A** — Las métricas de ejecución muestran presión de memoria y escritura a disco.

### Documentación oficial
- [Bundles](https://docs.databricks.com/aws/en/dev-tools/bundles/reference)
- [CLI](https://docs.databricks.com/aws/en/dev-tools/cli/bundle-commands)

## PARTE 6 - RETO FINAL
Añade una tarea de calidad que bloquee Gold, un trigger apropiado y un parámetro de fecha. Provoca un fallo, repara y entrega evidencias de dev/test separados.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
